In [1]:
import sys
sys.path.append("../..")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from src.cuantizacion import ajustar_cuantizador, transformar

In [2]:

x = [10, 12, 15, 21, 24, 31, 35, 40]
k = 4
amplitud = (max(x) - min(x)) / k
print("amplitud:", amplitud)          # 7.5
print("limites esperados:", [10 + i * amplitud for i in range(k + 1)])   # 10, 17.5, 25, 32.5, 40
print("codigos esperados:", [0, 0, 0, 1, 1, 2, 3, 3])

amplitud: 7.5
limites esperados: [10.0, 17.5, 25.0, 32.5, 40.0]
codigos esperados: [0, 0, 0, 1, 1, 2, 3, 3]


In [3]:
m = ajustar_cuantizador(x, "uniforme", k, columna="demo")
print(m)
codigos = transformar(x, m)
print(codigos, codigos.dtype)

Cuantizador(estrategia='uniforme', n_bins_solicitado=4, n_bins_real=4, limites=array([10. , 17.5, 25. , 32.5, 40. ]), representantes={}, columna='demo', n_ajuste=8)
[0 0 0 1 1 2 3 3] int16


In [4]:
def intervalo(c, modelo):
    if c < 0:
        return "NaN"
    a, b = modelo.limites[c], modelo.limites[c + 1]
    cierre = "]" if c == modelo.n_bins_real - 1 else ")"
    return f"[{a:g}, {b:g}{cierre}"

tabla = pd.DataFrame({
    "Valor": x,
    "Intervalo": [intervalo(c, m) for c in codigos],
    "Código": codigos,
})
tabla

,Valor,Intervalo,Código
0,10,"[10, 17.5)",0
1,12,"[10, 17.5)",0
2,15,"[10, 17.5)",0
3,21,"[17.5, 25)",1
4,24,"[17.5, 25)",1
5,31,"[25, 32.5)",2
6,35,"[32.5, 40]",3
7,40,"[32.5, 40]",3


In [5]:
import time
rng = np.random.default_rng(0)
x5 = rng.normal(size=5_000_000)

t0 = time.perf_counter()
m5 = ajustar_cuantizador(x5, "uniforme", 50)
t1 = time.perf_counter()
cod5 = transformar(x5, m5)
t2 = time.perf_counter()

print(f"ajuste {t1 - t0:.3f} s | transformar {t2 - t1:.3f} s | códigos {cod5.nbytes / 1e6:.1f} MB")

ajuste 0.019 s | transformar 0.167 s | códigos 10.0 MB
